# 05 - Async, limites y BaseModel

Este notebook muestra funciones asincronicas alrededor de llamadas LLM. El caso de uso es evaluar varias respuestas con limite de concurrencia y limite de texto generado, validando cada resultado con Pydantic.

Igual que en el notebook 04, aqui usamos `invoke_structured()` para enviar el `BaseModel` al SDK de OpenAI. En modo demo, validamos diccionarios locales con `model_validate()`.

In [ ]:
import asyncio
from pydantic import BaseModel, Field

from llmkit.config.settings import get_settings
from llmkit.llms import LLMFactory, build_messages

get_settings.cache_clear()
settings = get_settings()

MODEL_ID = settings.default_model
RUN_LIVE_LLM_CALLS = False
MAX_CONCURRENT_LLM_CALLS = 2
MAX_FEEDBACK_CHARS = 180

print("Model:", MODEL_ID)

In [ ]:
class AnswerMetric(BaseModel):
    name: str
    score: float = Field(ge=1, le=5)
    note: str = Field(max_length=MAX_FEEDBACK_CHARS)

class AsyncAnswerEval(BaseModel):
    question_id: str
    feedback: str = Field(max_length=MAX_FEEDBACK_CHARS)
    metrics: list[AnswerMetric]
    final_score: float = Field(ge=1, le=5)

class AsyncEvalBatch(BaseModel):
    results: list[AsyncAnswerEval]

AsyncAnswerEval.model_json_schema()

## Datos de entrada

Cada item representa una respuesta candidata. El flujo async permite procesarlas con control de concurrencia. El schema de arriba corresponde a cada evaluacion individual; el batch se arma localmente despues de validar cada respuesta.

In [ ]:
questions = [
    {
        "question_id": "q1",
        "question": "What is the practical role of an eval dataset?",
        "answer": "It gives repeatable questions and reference expectations so prompt changes can be compared.",
    },
    {
        "question_id": "q2",
        "question": "Why limit generated text in an evaluator?",
        "answer": "Short feedback is easier to scan and cheaper to store across many test cases.",
    },
    {
        "question_id": "q3",
        "question": "When should async be used?",
        "answer": "Use it when multiple independent calls can wait on IO at the same time.",
    },
]

questions

In [ ]:
def trim_text(text: str, max_chars: int = MAX_FEEDBACK_CHARS) -> str:
    text = " ".join(text.split())
    return text if len(text) <= max_chars else text[: max_chars - 3].rstrip() + "..."

def build_eval_prompts(item: dict) -> tuple[str, str]:
    system = (
        "You are a strict answer evaluator. Return only valid JSON matching this object contract: "
        "question_id is a string, feedback is a string, metrics is a list of objects "
        "with name, score, and note, and final_score is a number from 1 to 5. "
        f"Every feedback or note field must be under {MAX_FEEDBACK_CHARS} characters."
    )
    user = (
        f"Question id: {item['question_id']}\n"
        f"Question: {item['question']}\n"
        f"Candidate answer: {item['answer']}\n\n"
        "Evaluate accuracy and usefulness. Return an object, not a list, with question_id, feedback, metrics, and final_score."
    )
    return system, user

async def call_llm_eval(system: str, user: str) -> AsyncAnswerEval:
    if RUN_LIVE_LLM_CALLS:
        llm = LLMFactory.create(MODEL_ID)
        response = await asyncio.to_thread(
            llm.invoke_structured,
            system=system,
            user=user,
            schema=AsyncAnswerEval,
        )
        return response.parsed

    await asyncio.sleep(0.05)
    return AsyncAnswerEval.model_validate({
        "question_id": user.splitlines()[0].replace("Question id: ", ""),
        "feedback": trim_text("Clear and mostly complete answer for the intended evaluator example."),
        "metrics": [
            {"name": "accuracy", "score": 4.0, "note": trim_text("The answer matches the core idea.")},
            {"name": "usefulness", "score": 4.0, "note": trim_text("The explanation is concise and operational.")},
        ],
        "final_score": 4.0,
    })

async def evaluate_one(item: dict, semaphore: asyncio.Semaphore) -> AsyncAnswerEval:
    async with semaphore:
        system, user = build_eval_prompts(item)
        messages = build_messages(system, user)
        parsed = await call_llm_eval(system, user)
        parsed.feedback = trim_text(parsed.feedback)
        return parsed

async def evaluate_all(items: list[dict]) -> AsyncEvalBatch:
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_LLM_CALLS)
    tasks = [evaluate_one(item, semaphore) for item in items]
    results = await asyncio.gather(*tasks)
    return AsyncEvalBatch(results=results)

## Ejecutar batch async

En Jupyter se puede usar `await` directo en una celda. El semaforo limita cuantas llamadas LLM corren al mismo tiempo.

In [ ]:
batch = await evaluate_all(questions)
batch

In [ ]:
for result in batch.results:
    print(f"{result.question_id}: final_score={result.final_score:.1f} feedback={result.feedback}")
    for metric in result.metrics:
        print(f"  - {metric.name}: {metric.score:.1f} | {metric.note}")